# Sponsor synonym-rate scoping — stratified, random-order

**Question:** before building a sponsor-canonicalization *oracle* (ADR 0001,
Axis B″), measure how common synonyms are in the ~37.5k normalized unique sponsor
names — **broken out by sponsor type** (industry vs academic/institute vs
unknown), since these likely differ.

**Method (assumption-free except "synonyms may exist").** Within each stratum,
draw `X` independent random samples of `N` names **in random order (NOT sorted —
sorting would inject lexical adjacency and bias toward lexical synonyms)**. A
frontier LLM returns synonym groups within each sample via structured output.
The only inference is a Horvitz–Thompson estimate of total synonym *pairs* per
stratum: `P̂ = (observed pairs)/(X·π₂)`, `π₂ = n(n-1)/[Nₛ(Nₛ-1)]` — no
group-structure assumption.

**Why stratify:** answers industry-vs-institute directly; each stratum is smaller
than the whole corpus, so *f* is larger and signal per sample is higher.
**Limitation:** within-stratum sampling misses *cross-stratum* synonyms, but only
0.4% of names are class-ambiguous, so leakage is small.

**Cost:** ~`len(strata)·X` calls (~60) of ~15k in / ~2k out — under ~$10 on a
frontier model. Needs `ANTHROPIC_API_KEY` in `../.env`.

In [ ]:
import os, sys, json, time, random, math, collections
from math import comb
import numpy as np
import pandas as pd
from dotenv import load_dotenv

sys.path.insert(0, "..")
load_dotenv(os.path.join("..", ".env"))

from config.settings import get_duckdb_connection
from src.transform.normalize_sponsors import normalize_sponsor_name
import anthropic

# ---------------- parameters ----------------
N_SAMPLE   = 2000
X_SAMPLES  = 20
RUN_STRATA = ["INDUSTRY", "OTHER", "UNKNOWN"]   # big enough for real sampling at N=2000
MODEL      = "claude-opus-4-8"                  # frontier reasoner for MEASUREMENT; edit freely
MAX_TOKENS = 8192
SEED       = 42
OUT_PATH   = "sponsor_synonym_stratified_results.json"

random.seed(SEED); np.random.seed(SEED)
print(f"N_SAMPLE={N_SAMPLE}  X_SAMPLES={X_SAMPLES}  strata={RUN_STRATA}  MODEL={MODEL}")

## 1. Build the corpus, partitioned by modal `agency_class`

In [ ]:
con = get_duckdb_connection(read_only=True)
rows = con.execute("SELECT name, agency_class, nct_id FROM raw.sponsors WHERE name IS NOT NULL").fetchall()
con.close()

trials  = collections.defaultdict(set)
classes = collections.defaultdict(collections.Counter)
for nm, cls, nct in rows:
    k = normalize_sponsor_name(nm)
    if k:
        trials[k].add(nct)
        classes[k][cls] += 1

modal_class = {k: c.most_common(1)[0][0] for k, c in classes.items()}
trial_count = {k: len(v) for k, v in trials.items()}

strata = collections.defaultdict(list)
for k, cls in modal_class.items():
    strata[cls].append(k)
for s in strata:
    strata[s].sort()

print("stratum sizes (distinct normalized names):")
for cls, names in sorted(strata.items(), key=lambda x: -len(x[1])):
    flag = "  <- run" if cls in RUN_STRATA else ""
    print(f"  {cls:<12} {len(names):,}{flag}")

## 2. The LLM judge (conservative, structured output, **random order**)

In [ ]:
API_KEY = os.environ.get("ANTHROPIC_API_KEY", "")
assert API_KEY, "Add ANTHROPIC_API_KEY to ../.env before running this cell."
client = anthropic.Anthropic(api_key=API_KEY, max_retries=5)

SYNONYM_TOOL = {
    "name": "report_synonym_groups",
    "description": "Report groups of input names that refer to the SAME real-world organization.",
    "input_schema": {
        "type": "object",
        "properties": {
            "groups": {
                "type": "array",
                "description": "Each element is a list of >=2 input names referring to one organization.",
                "items": {"type": "array", "items": {"type": "string"}},
            }
        },
        "required": ["groups"],
    },
}

SYSTEM = ("You are an expert at clinical-trial sponsor/organization name "
          "disambiguation. You decide which raw organization-name strings refer "
          "to the SAME real-world organization.")

INSTRUCTIONS = (
    "Below is a list of organization names taken from clinical-trial records, in "
    "random order. Some may be variants of the SAME real-world organization: "
    "spelling, punctuation, word order, abbreviation/acronym vs full name, "
    "language/transliteration, or legal-entity wording. Read the WHOLE list and "
    "group the names that refer to the same organization.\n\n"
    "Rules:\n"
    "- Only form groups of 2 or more names; never list singletons.\n"
    "- Be CONSERVATIVE: if not confident two names are the same organization, do "
    "NOT group them. Distinct organizations sharing only a city or a generic word "
    "(e.g. 'University Hospital', 'Ministry of Health') are NOT the same.\n"
    "- Treat a parent and an explicit subsidiary/branch of the same company as the "
    "same organization; two genuinely different companies are not.\n"
    "- Copy names EXACTLY (verbatim) so they can be matched back.\n"
    "Return your answer by calling report_synonym_groups."
)

def find_synonym_groups(sample_names):
    listing = "\n".join(f"{i+1}. {n}" for i, n in enumerate(sample_names))
    msg = client.messages.create(
        model=MODEL, max_tokens=MAX_TOKENS, system=SYSTEM,
        tools=[SYNONYM_TOOL],
        tool_choice={"type": "tool", "name": "report_synonym_groups"},
        messages=[{"role": "user", "content": INSTRUCTIONS + "\n\n" + listing}],
    )
    groups = []
    for block in msg.content:
        if block.type == "tool_use" and block.name == "report_synonym_groups":
            groups = block.input.get("groups", []); break
    lut = {n.lower(): n for n in sample_names}
    clean = []
    for g in groups:
        seen = []
        for nm in g:
            k = str(nm).strip().lower()
            if k in lut and lut[k] not in seen:
                seen.append(lut[k])
        if len(seen) >= 2:
            clean.append(seen)
    return clean, (msg.usage.input_tokens, msg.usage.output_tokens)

## 3. Run X samples per stratum (random order, no sort)

In [ ]:
results = []
tin = tout = 0
t0 = time.time()
for cls in RUN_STRATA:
    pool = strata[cls]
    n = min(N_SAMPLE, len(pool))
    for s in range(X_SAMPLES):
        sample = random.sample(pool, n)          # RANDOM ORDER — deliberately not sorted
        try:
            groups, (ti, to) = find_synonym_groups(sample)
        except Exception as e:
            print(f"[{cls}] sample {s+1}: ERROR {type(e).__name__}: {e}"); continue
        tin += ti; tout += to
        pairs     = sum(comb(len(g), 2) for g in groups)
        in_groups = sum(len(g) for g in groups)
        results.append({"stratum": cls, "n_total": len(pool), "n_sample": n,
                        "sample": s, "n_groups": len(groups),
                        "names_in_groups": in_groups, "pairs": pairs,
                        "group_sizes": [len(g) for g in groups], "groups": groups})
    done = [r for r in results if r["stratum"] == cls]
    mp = np.mean([r["pairs"] for r in done]) if done else 0
    print(f"[{cls}] {len(done)} samples done, mean pairs/sample={mp:.1f}")

print(f"\ntotal time {time.time()-t0:.0f}s | tokens in={tin:,} out={tout:,}")
json.dump(results, open(OUT_PATH, "w"), indent=2)
print(f"saved -> {OUT_PATH}")

## 4. Eyeball quality per stratum

In [ ]:
for cls in RUN_STRATA:
    ex = [g for r in results if r["stratum"] == cls for g in r["groups"]][:10]
    print(f"--- {cls} (sample groups) ---")
    for g in ex:
        print("   ", "  |  ".join(g))
    print()

## 5. Per-stratum estimate — Horvitz–Thompson pairs + prevalence

`P̂ = pairs/(X·π₂)` is unbiased for total synonym pairs in the stratum (no
structural assumption). Prevalence uses the empirically observed names-per-pair
ratio (≈2 under pair-dominance) and is cross-checked against the de-thinned raw
rate `raw_rate / f`. Both are **floors** (LLM recall < 1).

In [ ]:
df = pd.DataFrame(results)
rng = np.random.default_rng(SEED)
table = []
for cls in RUN_STRATA:
    d = df[df["stratum"] == cls]
    if d.empty: continue
    n, Nt = int(d["n_sample"].iloc[0]), int(d["n_total"].iloc[0])
    pi2 = (n*(n-1))/(Nt*(Nt-1)); f = n/Nt
    pc = d["pairs"].values; X = len(d)
    P_hat = pc.sum()/(X*pi2)
    boot = [rng.choice(pc, size=X, replace=True).sum()/(X*pi2) for _ in range(5000)]
    lo, hi = np.percentile(boot, [2.5, 97.5])
    npp = d["names_in_groups"].sum()/max(d["pairs"].sum(), 1)     # names per pair (~2 if disjoint)
    names_syn = npp * P_hat
    prev = names_syn / Nt
    raw_rate = d["names_in_groups"].sum()/(X*n)
    prev_raw = raw_rate / f                                       # cross-check
    sizes = collections.Counter(k for r in d.to_dict("records") for k in r["group_sizes"])
    table.append({"stratum": cls, "corpus": Nt, "f": round(f,3),
                  "mean_pairs/smp": round(pc.mean(),1),
                  "P_hat_pairs": int(P_hat), "CI": f"{int(lo)}-{int(hi)}",
                  "names_w_syn": int(names_syn), "prevalence": f"{prev:.1%}",
                  "prev_xcheck": f"{prev_raw:.1%}",
                  "sizes": dict(sorted(sizes.items()))})
pd.set_option("display.width", 200, "display.max_columns", 20)
print(pd.DataFrame(table).to_string(index=False))

## 6. Scope decision (per stratum)

Compare INDUSTRY vs OTHER (institute) vs UNKNOWN:

- If **INDUSTRY prevalence is low** but **OTHER is high**, the oracle's value is
  concentrated in academic/hospital names — tier the build to spend there, and
  treat industry with lighter machinery (or ROR, which covers companies better).
- If **both are high**, the full Axis B″ oracle is warranted across the board.
- Pair-dominance (size histogram in the table) keeps clustering simple; presence
  of larger groups in any stratum flags chaining risk to guard.

All numbers are floors (random-order recall < 1); a stratum that already reads
"common" at the floor is conclusively common.